# 02 - Train MiniConvNet (the single v3 architecture, all lesson-2 fixes on)

**What this notebook does**: trains **one** MiniConvNet architecture - the ~0.5M-parameter Flatten
reading - on the `faithful` split (paper-comparable) and then on the `clean` split (robustness
experiment). It also measures the trained model's file size on disk to settle the paper's "~6MB"
claim (LESSON 9).

**What must already exist**: the split CSVs from notebook 00.

## The architecture decision (LESSON 3)

The paper's layer list (`GAP -> Dense(64) -> Dropout -> Dense(4)`) reaches only ~106K parameters and
contradicts the paper's own "~0.5M params, ~6MB". Both readings cannot be right. v2 built both, split
its compute two ways, and produced a mediocre result for each. **v3 commits to the ~0.5M reading and
builds only that.**

## The five anti-collapse measures, all on by default (LESSON 2)

v2's unresolved bug was *partial collapse*: the model predicted only 2 of 4 classes across every
configuration tried. Rather than testing remedies one at a time - unaffordable on CPU - all five are
applied together from the first run:

| # | Measure | Where |
|---|---|---|
| A | `LeakyReLU(alpha=0.1)` everywhere, **not** ReLU | `models.build_miniconvnet` |
| B | Wide 64-unit bottleneck instead of v2's `Dense(16)`, bought with one extra pooling layer | `models.build_miniconvnet` |
| C | He (`he_normal`) initialisation, not Keras's Glorot default | `models.build_miniconvnet` |
| D | `ReduceLROnPlateau(val_loss, patience=5, factor=0.5)` | `train_utils.make_callbacks` |
| E | `CategoricalCrossentropy(label_smoothing=0.05)` | `train_utils.compile_model` |

Plus the LESSON 1 fix that already worked: `Adam(lr=1e-4, clipnorm=1.0)`.

**Collapse detection runs on this first run, not after the fact** (LESSON 4), and raw predictions are
saved for every run (LESSON 5).

**CPU cost**: two training runs. Section 3 measures a real epoch and reports an estimate **before**
either starts - if it exceeds 20 minutes, stop and agree the cost first.

**What "looks right"**: validation accuracy clearly above chance (0.25) within the first ~10 epochs,
a loss curve that is not a flat line, and a collapse check that reports **all four classes predicted**.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

from src.config import *
from src.data_utils import resolve_data_root

ensure_dirs()
print('data root:', resolve_data_root())
print('models dir:', MODELS_DIR)
print()
print('anti-collapse settings actually in force (LESSON 2):')
for k, v in describe_anti_collapse_settings().items():
    print(f'  {k}: {v}')

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

from src.data_utils import load_split, make_split_datasets, split_counts
from src.models import (build_miniconvnet, count_params, check_param_budget,
                        architecture_summary, flatten_width)
from src.train_utils import (set_global_seeds, compute_report, compile_model, optimizer_summary,
                             class_weights_for, make_callbacks, make_epoch_timer, save_history,
                             plot_history, final_epoch_summary, run_name_for, checkpoint_path,
                             estimate_training_time, dead_unit_report,
                             measure_model_file_sizes, verdict_on_size_claim)
from src.evaluate_utils import (predict, compute_metrics, detect_collapse, print_collapse_report,
                                plot_confusion_matrix, confusion, per_class_report,
                                tumor_vs_subtype_breakdown, interpret_breakdown,
                                save_predictions, result_row_from_metrics, record_experiment,
                                load_results)

set_global_seeds(SEED)
for k, v in compute_report().items():
    print(f'{k}: {v}')
print()
print('epoch budget:', EPOCHS_MINICONVNET, '| early stopping patience:', EARLY_STOPPING_PATIENCE)

## 1. The model, and proof it hits the parameter budget

The whole architecture argument rests on the parameter count, so it is checked by code rather than
trusted to arithmetic in a comment.

**Looks right**:
- total parameters ~499,172, i.e. within a few tenths of a percent of the paper's ~0.5M;
- flatten width **6272** (7x7x128), not 25088 - that is the extra pooling layer doing its job;
- bottleneck **64** units, not 16;
- every activation layer a `LeakyReLU`.

In [ ]:
model = build_miniconvnet()
model.summary()

In [ ]:
print(count_params(model))
print()
budget = check_param_budget(model)
print()
for k, v in architecture_summary(model).items():
    print(f'  {k}: {v}')
print()
print('flatten width:', flatten_width(model),
      '(v2 used 25088 -> a 16-unit bottleneck; the extra pool cuts this 4x)')
assert budget['within_tolerance'], 'model is not within 5% of the paper\'s ~0.5M parameters'

## 2. Data

MiniConvNet trains with label smoothing, so **the datasets must be one-hot** (`one_hot=True`).
`compile_model()` prints which target format it expects; if the two disagree, training fails
immediately rather than silently learning the wrong thing.

**Looks right**: `faithful` ~613/72/315 train/val/test, `clean` smaller with a printed class-weight
dict (LESSON 7).

In [ ]:
def prepare(split_variant):
    sdf = load_split(split_variant)
    train_ds, val_ds, test_ds, frames = make_split_datasets(sdf, one_hot=True)
    cw = class_weights_for(split_variant, frames['train']['label'].values)
    print(f'--- {split_variant} ---')
    print(split_counts(sdf).to_string())
    print('class_weight:', {CLASS_NAMES[k]: round(v, 3) for k, v in cw.items()} if cw
          else 'None (by design for this split)')
    return {'df': sdf, 'train_ds': train_ds, 'val_ds': val_ds, 'test_ds': test_ds,
            'frames': frames, 'class_weight': cw, 'split_variant': split_variant}

data_faithful = prepare('faithful')
print()
data_clean = prepare('clean')

## 3. Time estimate before training anything (CPU constraint)

This trains a throwaway model for one real epoch, measures it, and extrapolates. The probe model is
discarded, so the real run still starts from a clean initialisation.

**Read the output before running section 4.** If it prints the over-threshold warning, stop and agree
the cost rather than launching the run.

In [ ]:
est = estimate_training_time(
    model_fn=lambda: compile_model(build_miniconvnet(), verbose=False),
    train_ds=data_faithful['train_ds'], val_ds=data_faithful['val_ds'],
    planned_epochs=EPOCHS_MINICONVNET, n_runs=2,      # faithful + clean
    class_weight=None)

## 4. Run helper

One function used for both splits: build -> compile -> fit -> evaluate -> collapse-check -> save
predictions -> log. Kept in the notebook so it can be edited while debugging.

Note the order inside it: **raw predictions are saved before anything is derived from them**
(LESSON 5), so even a run that turns out to be broken leaves behind the evidence needed to diagnose
it without retraining.

In [ ]:
def train_and_evaluate(data, config_note, epochs=EPOCHS_MINICONVNET,
                       dropout_rate=DROPOUT_RATE, run_suffix=None):
    split_variant = data['split_variant']
    set_global_seeds(SEED)
    run_name = run_name_for('miniconvnet', split_variant, run_suffix)
    print('=' * 72)
    print('run:', run_name)

    model = build_miniconvnet(dropout_rate=dropout_rate)
    compile_model(model)
    print('params   :', count_params(model)['total_params'])
    print('optimizer:', optimizer_summary(model))

    timer = make_epoch_timer(verbose=1)
    history = model.fit(data['train_ds'], validation_data=data['val_ds'], epochs=epochs,
                        class_weight=data['class_weight'],
                        callbacks=make_callbacks(run_name, timer=timer), verbose=2)
    save_history(history, run_name, timer=timer)
    summary = final_epoch_summary(history, timer=timer)
    print()
    for k, v in summary.items():
        print(f'  {k}: {v}')
    print('checkpoint:', checkpoint_path(run_name))

    # --- evaluation -----------------------------------------------------
    y_true, y_pred, y_prob = predict(model, data['test_ds'])
    metrics = compute_metrics(y_true, y_pred, y_prob)

    # LESSON 5: evidence first, conclusions second.
    save_predictions(run_name, y_true, y_pred, y_prob,
                     meta={'split_variant': split_variant, 'config_note': config_note,
                           'activation': ACTIVATION, 'label_smoothing': LABEL_SMOOTHING,
                           'dropout_rate': dropout_rate})

    collapse = detect_collapse(history=history, kappa=metrics['cohen_kappa'],
                               mcc=metrics['mcc'], y_pred=y_pred)
    breakdown = tumor_vs_subtype_breakdown(y_true, y_pred)

    print('\ntest metrics:')
    for k, v in metrics.items():
        print(f'  {k}: {v:.4f}')
    print('\nconfusion matrix (an all-zero COLUMN = a class never predicted):')
    print(confusion(y_true, y_pred))
    print()
    print_collapse_report(collapse, run_name)
    print('\nper-class report:')
    print(per_class_report(y_true, y_pred).round(4))
    print('\n' + interpret_breakdown(breakdown))

    plot_history(history, run_name)
    plot_confusion_matrix(y_true, y_pred, run_name)

    record_experiment(result_row_from_metrics(
        model_name=run_name, metrics=metrics, collapse=collapse,
        arch_variant='miniconvnet_500k', split_variant=split_variant,
        params=count_params(model)['total_params'],
        epochs_trained=summary['epochs_trained'], n_runs=1, breakdown=breakdown,
        notes=(f"mean {summary.get('mean_seconds_per_epoch', 'NA')}s/epoch on CPU; "
               f"total {summary.get('total_minutes', 'NA')} min"),
        config_note=config_note))

    return {'run_name': run_name, 'model': model, 'history': history, 'timer': timer,
            'metrics': metrics, 'collapse': collapse, 'breakdown': breakdown,
            'summary': summary, 'y_true': y_true, 'y_pred': y_pred}

## 5. Run A - `faithful` split (the paper-comparable one)

This is the run the paper comparison rests on. Single-run numbers stay in `experiments_log.csv`; the
headline number is the 3-fold CV mean from notebook 03 (LESSON 10).

**Looks right**: `PASS - no collapse detected; all classes predicted at least once`. That single line
is what v2 never achieved.

In [ ]:
run_a = train_and_evaluate(
    data_faithful,
    config_note=('v3 MiniConvNet ~499K, faithful split, single run; LeakyReLU(0.1) + he_normal + '
                 'Dense(64) bottleneck + label_smoothing=0.05 + ReduceLROnPlateau + '
                 'Adam(1e-4, clipnorm=1.0)'))

## 6. Did the collapse mechanism actually go away? (LESSON 2 diagnostics)

The dead-unit probe measures the thing the architecture changes were meant to fix: units that never
activate positively on any sample. If a partial collapse ever returns, this says whether dying
activations are the cause or whether the problem lies elsewhere.

**Looks right**: a low dead fraction in `act_dense_1` (the 64-unit bottleneck). A dead fraction above
~50% there means the bottleneck is starved and classes are becoming unreachable - the v2 failure mode.

In [ ]:
dead = dead_unit_report(run_a['model'], data_faithful['test_ds'], batches=4)
print()
bottleneck = dead.get('act_dense_1')
if bottleneck:
    if bottleneck['dead_fraction'] > 0.5:
        print('WARNING: the bottleneck is majority-dead. If the run also partially collapsed, dying '
              'activations are the confirmed cause - report that, do not just try another seed.')
    else:
        print(f"Bottleneck is healthy ({bottleneck['dead_units']}/{bottleneck['units']} units never "
              'positive). If a partial collapse still occurred, dying units are NOT the explanation '
              'and the cause lies elsewhere (class separability, data, or capacity).')

## 7. The "~6MB" claim, measured (LESSON 9)

The paper states "~0.5M parameters, ~6MB", but ~499K float32 weights is only ~1.9MB. Adam keeps two
extra buffers per parameter (momentum + variance), so a checkpoint saved *with* the optimizer should
cost roughly 3x the weights - ~5.7MB, which is what "~6MB" would mean if the authors used Keras's
default `model.save()` rather than `save_weights()`.

This measures both instead of arguing about it. It needs the **trained** model from run A: Adam's
slot variables are created lazily, so an untrained model has no optimizer state and both files would
come out the same size for the wrong reason. `optimizer_slot_scalars_found` is the guard - if it
prints 0, the result is inconclusive, not confirmatory.

In [ ]:
size_check = measure_model_file_sizes(run_a['model'], run_name=run_a['run_name'])
print()
print(verdict_on_size_claim(size_check))
print()
print('Copy the two measured MB numbers into README lesson 9 verbatim - do not paraphrase them, '
      'and do not substitute the predicted values above for the measured ones.')

## 8. Run B - `clean` split (robustness experiment, **not** a replication)

The clean split removes the duplicate-driven train/test leakage, so accuracy here is expected to be
*lower* than on `faithful`. **That drop is the result, not a failure** - it is the measure of how much
of the paper-comparable number the leakage was worth.

Class weights are on for this split (LESSON 7).

In [ ]:
run_b = train_and_evaluate(
    data_clean,
    config_note=('v3 MiniConvNet ~499K, CLEAN split (deduplicated, leakage-free) - robustness '
                 'experiment, NOT a paper replication; class weights on; same anti-collapse '
                 'settings as the faithful run'))

## 9. Side-by-side summary

**Looks right**: two rows, both `status = ok`, both with `classes_predicted = 4`. Any row tagged
`INVALID_collapsed` or `INVALID_partial_collapse` must not be quoted anywhere - see the guidance
under the table.

In [ ]:
rows = []
for r in (run_a, run_b):
    rows.append({
        'run': r['run_name'],
        'split': r['run_name'].split('_')[-1],
        'accuracy': round(r['metrics']['accuracy'], 4),
        'f1_macro': round(r['metrics']['f1_macro'], 4),
        'cohen_kappa': round(r['metrics']['cohen_kappa'], 4),
        'tumor_vs_healthy': round(r['breakdown']['binary_tumor_vs_healthy_accuracy'], 4),
        'subtype_acc': round(r['breakdown']['subtype_accuracy_all_tumors'], 4),
        'classes_predicted': r['collapse']['details'].get('n_predicted_classes'),
        'epochs': r['summary']['epochs_trained'],
        'min': r['summary'].get('total_minutes'),
        'status': r['collapse']['status'],
    })
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

drop = run_a['metrics']['accuracy'] - run_b['metrics']['accuracy']
print(f'\nfaithful - clean accuracy drop: {drop:+.4f} '
      '(how much of the paper-comparable number the duplicate leakage was worth)')

bad = summary[summary['status'] != VALID_TAG]
if len(bad):
    print('\n!!! invalid run(s) - do NOT report these numbers:')
    print(bad[['run', 'status']].to_string(index=False))
    print('\nWith all five lesson-2 measures already applied, a partial collapse here is a FINDING, '
          'not a cue to start tuning. Check the dead-unit probe in section 6, write down what it '
          'says, and report the plateau with the confusion-matrix evidence (see the README\'s '
          '"what improving accuracy means, and its limit").')
else:
    print('\nBoth runs predicted all four classes and passed the collapse check - v2\'s partial '
          'collapse did not recur.')

In [ ]:
log = load_results('experiment')
print(f'experiments_log.csv now has {len(log)} rows')
print(log[['model', 'split_variant', 'accuracy', 'binary_tumor_acc', 'subtype_acc',
           'status']].tail(4).to_string(index=False))
print('\nnext: 03_cross_validation.ipynb (which produces the canonical MiniConvNet number)')